<b><font size="6" color="#E8800A">Missing values, invalid values and outliers</font></b><br>

This notebook purpouse is to handle missing values and outliers.

We imported the necessary libraries and the data, furthermore we divided the columns again into the same five different categories.

In [2]:
import sys
import warnings
from collections import namedtuple
from functools import partial
from pathlib import Path

# course_helpers.py sits in this folder, next to the notebook.
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_theme(style="whitegrid")
from math import ceil          # cell 22 crashes without it
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit, train_test_split
from sklearn.exceptions import ConvergenceWarning
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import (
    MinMaxScaler, OneHotEncoder, RobustScaler, StandardScaler,
)

# course_helpers.py is the course toolbox: one file, kept beside every
# notebook, holding what is the same every week. It contains no machine
# learning, only the two plot colours and CleaningLog, a small recorder for the
# decisions made below. It is short; open it.
from course_helpers import PLOT_BLUE, PLOT_ORANGE, CleaningLog

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)  # For reproducibility
FIGSIZE = (10, 6)

# One warning class is silenced here, on purpose. Read the next cell before
# deciding whether that is ever a reasonable thing to do.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [3]:
train_df = pd.read_csv("../data/train.csv", index_col="ID")
clients_df = pd.read_csv("../data/clientes.csv", index_col="customer_id")
test_df = pd.read_csv("../data/test.csv", index_col="ID")

In [4]:
train_df["purchase_date"] = pd.to_datetime(train_df["purchase_date"], format="%Y-%m-%d")

In [5]:
identifiers = ["customer_id"]
target = "repeat_purchase_90d"
date = ["purchase_date"]

numeric = [c for c in train_df.columns
           if c not in identifiers + [target] + date
           and pd.api.types.is_numeric_dtype(train_df[c])]
categorical = [c for c in train_df.columns
               if c not in identifiers + [target] + date + numeric]

<b><font size="6" color="#E8800A">Computing values from other columns</font></b><br>

Before changing any values, we create a copy of the original train dataset, for good practice.

In [7]:
train_copy = train_df.copy()

**Step 1:** For a better understanding of how missing values behave, we calculated how many of them each column has.

In [8]:
train_df.isna().sum()

Total a Pagar                       65
Total Bruto                         65
Total Liquido                       65
Total Imp_                          65
Total Portes                        65
Total Desc_ Global                  65
Total Desc_ Linha                   65
Custo                               65
N_ Detalhes                         65
invoice_count_current_purchase      65
repeat_purchase_90d                  0
days_since_previous_purchase       861
customer_tenure_days                65
prior_purchase_count                65
prior_total_spend                   65
prior_mean_purchase_value          862
prior_std_purchase_value          1113
prior_min_purchase_value           668
prior_max_purchase_value           668
prior_mean_gap_days               1308
prior_std_gap_days                1478
purchase_count_30d                  65
spend_30d                           65
purchase_count_90d                  65
spend_90d                           65
purchase_count_180d      

In [9]:
train_df[numeric+categorical].isna().all(axis=1).sum()

np.int64(65)

65 rows have missing values (NaN) in all columns except `customer_id` and `repeat_purchase_90d` (target). These 65 rows offer no information, beside customer id and if they bought again within 90 days, so we decided to drop them.

In [10]:
columns_null = numeric + categorical
train_copy = train_copy.dropna(subset=columns_null, how="all")
train_copy[columns_null].isna().sum()[lambda s: s > 0]

days_since_previous_purchase     796
prior_mean_purchase_value        797
prior_std_purchase_value        1048
prior_min_purchase_value         603
prior_max_purchase_value         603
prior_mean_gap_days             1243
prior_std_gap_days              1413
spend_365d                       195
payment_type                     195
salesperson                      194
transport_method                 196
warehouse                        195
dtype: int64

Now we only have 12 columns with missing values, instead of the original 36.

**Step 2:** The `prior_mean_gap_days`and `prior_mean_purchase_value` columns can both be computed using other columns. In the data exploration notebook we saw that they both had negative values. When we analysed the negative values we saw that two of the five columns had the same minimum (-4), so we decided to check the distinct negative values. We excluded `random_noise`because it is the only column where a negative value makes sense.

In [11]:
for c in ["days_since_previous_purchase", "prior_mean_gap_days", "prior_mean_purchase_value", "spend_365d"]:
    print(train_df.loc[train_df[c] < 0, c].value_counts())

days_since_previous_purchase
-4.0    16
Name: count, dtype: int64
prior_mean_gap_days
-4.0    16
Name: count, dtype: int64
prior_mean_purchase_value
-110.147097     1
-409.791667     1
-218.266392     1
-417.917143     1
-779.030000     1
-212.360622     1
-366.120000     1
-2478.658478    1
-693.010000     1
-757.480000     1
-305.930000     1
-504.580000     1
-357.193750     1
-162.482308     1
-883.510000     1
-885.097111     1
Name: count, dtype: int64
spend_365d
-3456.320000      1
-1230.720000      1
-167.180000       1
-753.510000       1
-2648.440000      1
-2169.910000      1
-1443.640000      1
-5133.040000      1
-14740.290000     1
-4196.900000      1
-1620.830000      1
-6795.470000      1
-159115.175535    1
-295.380000       1
-12638.050000     1
Name: count, dtype: int64


While in `prior_mean_purchase_value` and `spend_365d` all negative values are different, in the `days_since_previous_purchase` and `prior_mean_gap_days` columns all negative values are -4. This number appears to be a placeholder.

We then proceeded to the calculation of `prior_mean_purchase_value`, when `prior_purchase_count`is grater than 0, using the formula: $$ \text{prior mean purchase value} = \frac{\text{prior total spend}}{\text{prior purchase count}} $$
 
 And the calculation for `prior_mean_gap_days`, when `prior_purchase_count`is grater than 1 and `days_since_previous_purchase` is grater or equal to 0, using the formula: $$ \text{prior mean gap days} = \frac{\text{customer tenure days} - \text{days since previous purchase}}{\text{prior purchase count} -1 } $$



In [12]:
positive_count = train_copy.loc[train_copy["prior_purchase_count"] > 0, "prior_purchase_count"]
expected_mean = train_copy["prior_total_spend"] / positive_count

stored = train_copy["prior_mean_purchase_value"]
wrong = stored.notna() & expected_mean.notna() & ~np.isclose(stored, expected_mean, atol=0.01)
print("stored mean differs from total/count on", wrong.sum(), "rows")

train_copy.loc[train_copy["prior_purchase_count"] > 0, "prior_mean_purchase_value"] = expected_mean[train_copy["prior_purchase_count"] > 0]

stored mean differs from total/count on 78 rows


In [13]:
mean_formula = (train_copy['customer_tenure_days']-train_copy['days_since_previous_purchase'])/(train_copy['prior_purchase_count']-1)

stored_gap = train_copy["prior_mean_gap_days"]
wrong_gap = stored_gap.notna() & mean_formula.notna() & ~np.isclose(stored_gap, mean_formula, atol=0.01)
print("stored mean differs from total/count on", wrong_gap.sum(), "rows")

stored mean differs from total/count on 26 rows


In [16]:
#turning the -4 placeholder in Nan
train_copy['prior_mean_gap_days'] = train_copy['prior_mean_gap_days'].replace(-4, np.nan)

days = train_copy["days_since_previous_purchase"]
gap = train_copy["prior_mean_gap_days"]
tenure = train_copy["customer_tenure_days"]
count = train_copy["prior_purchase_count"]

rows_to_substitute = (count > 1) & (days >= 0)

train_copy.loc[rows_to_substitute, 'prior_mean_gap_days'] = (train_copy.loc[rows_to_substitute, 'customer_tenure_days']-train_copy.loc[rows_to_substitute, 'days_since_previous_purchase'])/(train_copy.loc[rows_to_substitute, 'prior_purchase_count']-1)

**Step 3:** We noticed before that the column `spend_365d` had some inconsistent values, inclunding negative values, and missing values. We decided to check any of those values could be filled or replaced by `prior_total_spend`. This applies to cases such as where `customer_tenure_days`is lower than 365 days or `prior_purchase_count` is the same as `purchase_count_365d`.

In [18]:
s = train_copy["spend_365d"]
inconsistent = s.notna() & ((s < train_copy["spend_180d"])
                            | (s > train_copy["prior_total_spend"] + 0.01))
print(inconsistent.sum())
train_copy.loc[inconsistent, "spend_365d"] = np.nan

c365 = train_copy["purchase_count_365d"]
all_in = c365.eq(train_copy["prior_purchase_count"])
none_in = c365.eq(0)
known = train_copy["spend_365d"].notna()
share_all = np.isclose(train_copy.loc[known & all_in, "spend_365d"],
                       train_copy.loc[known & all_in, "prior_total_spend"], atol=0.01).mean()
share_none = train_copy.loc[known & none_in, "spend_365d"].eq(0).mean()
print(round(share_all, 3), round(share_none, 3))
if share_all > 0.99:
    m = train_copy["spend_365d"].isna() & all_in
    train_copy.loc[m, "spend_365d"] = train_copy.loc[m, "prior_total_spend"]
if share_none > 0.99:
    m = train_copy["spend_365d"].isna() & none_in
    train_copy.loc[m, "spend_365d"] = 0.0


69
1.0 1.0


**Step 4:** For customers that have 0 prior purchases, it's normal for them to not have `days_since_previous_purchase`, `prior_mean_gap_days` and `prior_std_gap_days`, since they have never bought anything before the current purchase.

For customers with 1 prior purchase, they can have `days_since_previous_purchase`, but since they only purchased once before, they don't have `prior_mean_gap_days` because there's no second prior purchase to measure the gap. This also means there won't be a `prior_std_gap_days`.

For customers with 2 prior purchases, there is only 1 historical gap, so we can calculate `prior_mean_gap_days` but not the sample standard deviation, `prior_std_gap_days`.

So how do we fill these missing values?

We fill missing prior_std_gap_days with 0 for customers with 2 prior purchases, since they only have one historical gap. We also fill missing `prior_std_purchase_value` with 0 for customers with 1 prior purchase, since they only have one previous purchase value. Using 0 in these cases is a modelling convention.

The remaining missing values are left open at this stage and handled later by the imputation plan fitted within each training split.

In [19]:
# one value has no spread: a convention, recorded in the log
m = count.eq(2) & train_copy["prior_std_gap_days"].isna()
train_copy.loc[m, "prior_std_gap_days"] = 0.0
m = count.eq(1) & train_copy["prior_std_purchase_value"].isna()
train_copy.loc[m, "prior_std_purchase_value"] = 0.0

<b><font size="6" color="#E8800A">Cleaning decisions</font></b><br>

Every decision below (impossible values, repeated customers, missing values, outliers) is a choice between treatments, and each is settled by measurement: apply the treatment, fit a model, and score it on rows the treatment never saw. This cell builds the tools for that.

**The rule behind all of it:** anything learned from the data (categories, fill values, scalers) is learned from the training rows of each split and applied to the test rows. Computing it on the whole file first would give the test rows a say in their own score.

**Notes for reading the results**
- The same twenty splits, grouped by `customer_id`, score every treatment, so treatments are compared on identical held-out customers and the differences are paired.
- `customer_id` is an identifier. It is kept out of the model, and only used to group the splits.
- The solver may stop at its iteration cap on unscaled columns. The scores remain comparable between treatments, but the absolute F1 values are not final, and scaling is tested at the end of the notebook.

In [20]:
def fold_matrices(X_train, X_test, categorical, numeric):
    """Learn the categories from training rows and transform both halves."""
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    cat_train = encoder.fit_transform(X_train[categorical].astype("string"))
    cat_test = encoder.transform(X_test[categorical].astype("string"))

    num_train = X_train[numeric].to_numpy(dtype=float)
    num_test = X_test[numeric].to_numpy(dtype=float)

    return (np.hstack([cat_train, num_train]),
            np.hstack([cat_test, num_test]))


def fill_missing(train, test, plan):
    """Fill every missing value with imputers fitted on `train` alone.

    `plan` is a list of (columns, imputer) pairs, and any scikit-learn imputer
    works: each one is fitted on the training rows of its own columns and fills
    those columns in both halves, so the test rows never help compute a fill.
    """
    train, test = train.copy(), test.copy()
    for columns, imputer in plan:
        # scikit-learn reads np.nan as a gap but not pd.NA, the gap of a
        # nullable column such as the booleans, so every gap becomes np.nan.
        blocks = [half[columns].astype(object).fillna(np.nan).infer_objects()
                  for half in (train, test)]
        if isinstance(imputer, (KNNImputer, IterativeImputer)):
            # KNN measures distances between rows, and the model inside
            # IterativeImputer can be any regressor, many of them sensitive to
            # scale. So both work on standardised columns, and their fills go
            # back in the columns' own units.
            scale = StandardScaler().fit(blocks[0])
            imputer.fit(scale.transform(blocks[0]))
            filled = [scale.inverse_transform(imputer.transform(scale.transform(block)))
                      for block in blocks]
        else:
            imputer.fit(blocks[0])
            filled = [imputer.transform(block) for block in blocks]
        train[columns], test[columns] = filled
    return train, test


# A model cannot fit a missing value, so every comparison fills inside its own
# split. Until the missing-values section chooses a plan, that fill is the
# training median for the numbers and the training mode for the labels.
median_mode = [(numeric, SimpleImputer(strategy="median")),
               (categorical, SimpleImputer(strategy="most_frequent"))]

Scored = namedtuple("Scored", "f1 iterations")


def sem(values):
    """Standard error of the mean: how far it would move on new splits."""
    return values.std(ddof=1) / np.sqrt(len(values))


def held_out_f1(prepare, frame, splits, numeric, categorical, target="repeat_purchase_90d",
                scaler=None):
    """Score one treatment by F1 on each split's untouched test rows.

    `prepare(train, test)` applies the treatment, fitted on `train` only. The
    result holds the test-row F1 per split (`.f1`) and the solver's iterations
    (`.iterations`). `scaler`, when given, is fitted on the training half and
    applied to both.
    """
    f1, iterations = [], []
    for train_rows, test_rows in splits:
        train, test = prepare(frame.iloc[train_rows], frame.iloc[test_rows])
        a, b = fold_matrices(train, test, categorical, numeric)
        if scaler is not None:
            fitted = scaler.fit(a)
            a, b = fitted.transform(a), fitted.transform(b)
        model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
        model.fit(a, train[target])
        iterations.append(int(model.n_iter_[0]))
        f1.append(f1_score(test[target], model.predict(b)))
    return Scored(np.array(f1), np.array(iterations))


print(f"{len(numeric)} numeric columns, {len(categorical)} categorical,"
      f" {len(identifiers)} identifiers held out of the model")

29 numeric columns, 6 categorical, 1 identifiers held out of the model


In [21]:
splits = GroupShuffleSplit(n_splits=20, test_size=0.2, random_state=RANDOM_STATE)
repaired = train_copy
repaired_splits = list(splits.split(repaired, repaired["repeat_purchase_90d"],
                                    groups=repaired["customer_id"]))
print(f"{len(repaired_splits)} splits on {len(repaired):,} rows;")

20 splits on 6,469 rows;


In [22]:
repeat_floor = []

for training_rows, validation_rows in repaired_splits:
    y_valid = repaired.iloc[validation_rows]["repeat_purchase_90d"]

    predictions = np.ones(len(y_valid), dtype=int)

    score = f1_score(y_valid, predictions)
    repeat_floor.append(score)

print("Always-repeat F1:", np.mean(repeat_floor))

Always-repeat F1: 0.6945271078398532


We predicted a repeat purchase (1) for every row in each validation split and calculated the average F1 score (we obtained 0.6945). This shows that a score close to 0.69 can be achieved without learning any patterns. We will check whether ouur models get a better F1 score than simply predicting that every customer will buy again.

In [23]:
#o que acham disto?

without_noise = [c for c in numeric if c != "random_noise"]
plan_with = [(numeric, SimpleImputer(strategy="median")),
             (categorical, SimpleImputer(strategy="most_frequent"))]
plan_without = [(without_noise, SimpleImputer(strategy="median")),
                (categorical, SimpleImputer(strategy="most_frequent"))]
f_with = held_out_f1(partial(fill_missing, plan=plan_with), repaired, repaired_splits,
                     numeric, categorical).f1
f_without = held_out_f1(partial(fill_missing, plan=plan_without), repaired, repaired_splits,
                        without_noise, categorical).f1
diff = f_with - f_without
print(f"{diff.mean():+.4f} +/- {sem(diff):.4f}")

+0.0003 +/- 0.0007


<b><font size="5" color="#E8800A">Invalid values</font></b><br>

The columns `days_since_previous_purchase`and `spend_365` still have negative values, which is impossible. In the next cells we test several fixes for these values.

In [24]:
invalid_errors = {
    "days_since_previous_purchase": lambda values: values < 0,
    "spend_365d": lambda values: values < 0,
}
invalid_cols = list(invalid_errors)
numeric_invalid = ["days_since_previous_purchase", "spend_365d"]


def domain_flags(frame, cols, errors):
    """One boolean column per rule, True where that column's value is impossible."""
    return pd.concat([errors[column](frame[column]) for column in cols], axis=1)


def fix_invalid(frame, cols, errors, replacement):
    """Every impossible value in `cols` becomes `replacement`.

    `np.nan` blanks the cell, recording that the value is unknown; `0` clips it,
    claiming the value was meant to be zero. Which is right is measured below.
    """
    out = frame.copy()
    out[cols] = out[cols].mask(domain_flags(frame, cols, errors), replacement)
    return out

def treat_invalid(train, test, plan):
    """Apply each (columns, replacement) pair of `plan`, then fill."""
    for cols, replacement in plan:
        train = fix_invalid(train, cols, invalid_errors, replacement)
        test = fix_invalid(test, cols, invalid_errors, replacement)
    return fill_missing(train, test, median_mode)


def drop_invalid_rows(train, test):
    """Drop the offending rows from the TRAINING half, then fill the rest."""
    return fill_missing(
        train[~domain_flags(train, invalid_cols, invalid_errors).any(axis=1)],
        test, median_mode)


def flag_invalid_rows(train, test):
    """Leave the impossible value in place and record it in its own column."""
    train, test = (pd.concat([f, domain_flags(f, invalid_cols, invalid_errors)
                              .add_suffix(" out of domain")],
                             axis=1) for f in (train, test))
    return fill_missing(train, test, median_mode)


arms = {
    "blank the cell": partial(treat_invalid,
                              plan=[(invalid_cols, np.nan)]),
    "clip to zero": partial(treat_invalid,
                            plan=[(numeric_invalid, 0)]),
    "drop the row": drop_invalid_rows,
    "keep and flag": flag_invalid_rows,
}
# The flag arm adds columns, so its scorer needs the longer categorical list.
flag_columns = domain_flags(repaired, invalid_cols, invalid_errors).add_suffix(
    " out of domain")
flagged_categorical = categorical + list(flag_columns.columns)
invalid_scores = {
    name: held_out_f1(fn, repaired, repaired_splits, numeric,
                      flagged_categorical if name == "keep and flag" else categorical).f1
    for name, fn in arms.items()
}
control = invalid_scores["blank the cell"]

# One row per arm: its own mean, and its difference from the control arm
# measured on the same twenty splits.
invalid_board = {}
for name, scores in invalid_scores.items():
    difference = scores - control
    invalid_board[name] = {
        "F1": scores.mean(),
        "vs blank": difference.mean(),
        "paired SEM": sem(difference),
        "beats blank on": f"{int((difference > 0).sum())} of {len(difference)}",
    }
print(pd.DataFrame.from_dict(invalid_board, orient="index").round(4).to_string())

# What the drop arm is actually spending, which the F1 column alone does not say.
flagged_rows = flag_columns.any(axis=1)
print(f"\nrows carrying an impossible value: {flagged_rows.sum():,}"
      f" of {len(repaired):,}"
      f"   win rate {repaired.loc[flagged_rows, 'repeat_purchase_90d'].mean():.4f}"
      f" against a base rate of {repaired['repeat_purchase_90d'].mean():.4f}")

                    F1  vs blank  paired SEM beats blank on
blank the cell  0.6910    0.0000      0.0000        0 of 20
clip to zero    0.6911    0.0001      0.0007        8 of 20
drop the row    0.6922    0.0012      0.0009       12 of 20
keep and flag   0.6907   -0.0003      0.0008        8 of 20

rows carrying an impossible value: 16 of 6,469   win rate 0.3125 against a base rate of 0.5597


No fix beat "blank the cell" because every SEM is within one standard error. Therefore we choose "blank the cell", which gives us an F1 score of 0.6910.

In [25]:
chosen_invalid_fix = "blank the cell"

masked = fix_invalid(repaired, invalid_cols, invalid_errors, np.nan)
print(f"{train_copy.shape[0]:,} rows, {masked.isna().sum().sum():,} null cells"
      f" once the invalid values are blanked, nothing filled or removed yet")

6,469 rows, 6,276 null cells once the invalid values are blanked, nothing filled or removed yet


<b><font size="5" color="#E8800A">Identifier</font></b><br>

In the data exploration notebook we saw that all the purchases (rows) from our data belonged only to 610 different customers, with 453 of them having more than one row associated. Having the same customer in both sides of the split favours the score, so the cells from this section are dedicated to understanding the best way to handle the customers.

In [26]:
def deduplicate(frame, key, choose):
    """Keep one row per `key`; `choose` takes one key's rows and names the survivor."""
    keep = frame.groupby(key, sort=False).apply(choose)
    return frame[frame.index.isin(keep)]


# Each `choose` returns the index label of the row that survives. The rule has
# to be deterministic, so `most complete` breaks its own ties on the earliest
# RecordID.
tiebreaks = {
    "earliest ID": lambda rows: rows.index.min(),
    "latest purchase_date": lambda rows: rows["purchase_date"].idxmax(),
    "most complete row": lambda rows: rows.isna().sum(axis=1).sort_index()
                                          .idxmin(),
}
# Five rules with no argument behind them at all, as a yardstick.
for seed in range(5):
    tiebreaks[f"arbitrary (seed {seed})"] = (
        lambda rows, s=seed: rows.sample(1, random_state=s).index[0])

candidates = {name: deduplicate(masked, "customer_id", choose)
              for name, choose in tiebreaks.items()}
candidates["keep every row"] = masked


def train_on(train, test, candidate):
    """Swap in the candidate's rows, minus every athlete in the test half, then fill.

    Each candidate keeps a different 4,000 rows, so all of them are scored on ONE
    set of test rows, drawn on the earliest-RecordID frame.
    """
    train = candidate[~candidate["customer_id"].isin(test["customer_id"])]
    return fill_missing(train, test, median_mode)


reference = candidates["earliest ID"]
reference_splits = list(splits.split(reference, reference["repeat_purchase_90d"],
                                     groups=reference["customer_id"]))
tiebreak_scores = {
    name: held_out_f1(partial(train_on, candidate=frame), reference,
                      reference_splits, numeric, categorical).f1
    for name, frame in candidates.items()}
earliest_score = tiebreak_scores["earliest ID"]

tiebreak_board = {}
for name, frame in candidates.items():
    scores = tiebreak_scores[name]
    difference = scores - earliest_score
    tiebreak_board[name] = {
        "rows": len(frame),
        "missing values": int(frame.isna().sum().sum()),
        "win rate": frame["repeat_purchase_90d"].mean(),
        "F1": scores.mean(),
        "vs earliest": difference.mean(),
        "paired SEM": sem(difference),
    }
print(pd.DataFrame.from_dict(tiebreak_board, orient="index").round(4).to_string())

                      rows  missing values  win rate      F1  vs earliest  paired SEM
earliest ID            607            2003    0.2636  0.3781       0.0000      0.0000
latest purchase_date   607            1376    0.0675  0.1256      -0.2525      0.0168
most complete row      607            1309    0.2142  0.4190       0.0409      0.0146
arbitrary (seed 0)     607            2104    0.2900  0.4139       0.0358      0.0153
arbitrary (seed 1)     607            2081    0.2850  0.4247       0.0466      0.0159
arbitrary (seed 2)     607            1943    0.2932  0.4311       0.0530      0.0229
arbitrary (seed 3)     607            1987    0.2537  0.3805       0.0023      0.0160
arbitrary (seed 4)     607            1866    0.2817  0.4267       0.0486      0.0162
keep every row        6469            6276    0.5597  0.4867       0.1086      0.0181


We keep all rows

In [29]:
chosen_tiebreak = "keep every row"

# Select, do not re-sort: the rows come back in the order the file had them,
# which is what keeps this step from quietly reordering the whole dataset.
deduped = masked
deduped_splits = list(splits.split(repaired, repaired["repeat_purchase_90d"],
                                    groups=repaired["customer_id"]))
print(f"{len(masked):,} rows -> {len(deduped):,} rows"
      f"  ({len(masked) - len(deduped):,} removed),"
      f" {deduped.isna().sum().sum():,} missing values still open")

6,469 rows -> 6,469 rows  (0 removed), 6,276 missing values still open


<b><font size="5" color="#E8800A">Missing values</font></b><br>

In the previous code cell, we can see that there are still many missing values ir our dataset (6276). In this section we will test which is the best method to impute them.

We started by identifying the columns in which a zero means something. In this case a customer that made 0 purchases didn't buy anything and a customer that spent 0 hasn't spent any money. We kept the columns where this rule applies in the list `zero_means_none` defined in the cell below.

In [30]:
zero_means_none = [
    "purchase_count_30d", "purchase_count_90d", "purchase_count_180d", "purchase_count_365d",
    "prior_purchase_count",
    "spend_30d", "spend_90d", "spend_180d", "spend_365d",
    "prior_total_spend",
    "Total Portes", "Total Desc_ Global", "Total Desc_ Linha",
]
zero_means_none = [c for c in zero_means_none if c in numeric]

In [ ]:
class ProportionalImputer:
    """Fill each missing label with one drawn from its column's training labels,
    in their training shares.

    scikit-learn has no such imputer, and `fill_missing` needs only `fit` and
    `transform`, so a short class is enough.
    """

    def __init__(self, random_state=None):
        self.random_state = random_state

    def fit(self, frame):
        self.shares = {c: frame[c].value_counts(normalize=True) for c in frame}
        return self

    def transform(self, frame):
        draws = np.random.default_rng(self.random_state)
        frame = frame.copy()
        for column, shares in self.shares.items():
            gaps = frame[column].isna()
            if gaps.any():
                frame.loc[gaps, column] = draws.choice(
                    shares.index, size=gaps.sum(), p=shares.to_numpy())
        return frame


# One imputer per fill. "own level" gives a missing label a category of its
# own instead of the common one.
fills = {"median": SimpleImputer(strategy="median"),
         "mean": SimpleImputer(strategy="mean"),
         "mode": SimpleImputer(strategy="most_frequent"),
         "zero": SimpleImputer(strategy="constant", fill_value=0),
         "own level": SimpleImputer(strategy="constant", fill_value="(missing)"),
         "proportional": ProportionalImputer(random_state=RANDOM_STATE),
         # Two that read the other numeric columns to fill each one.
         "KNN (k=5)": KNNImputer(n_neighbors=5),
         "iterative": IterativeImputer(random_state=RANDOM_STATE, max_iter=10)}

# Which numeric columns a mean would misrepresent, by the same skew measured
# in the distributions section, rather than by re-deriving it from dtype.
column_skew = deduped[numeric].skew()
skewed = column_skew[column_skew.abs() > 1].index.tolist()
symmetric = column_skew[column_skew.abs() <= 1].index.tolist()

plans = {
    "median / mode": [(numeric, fills["median"]), (categorical, fills["mode"])],
    "median / own level": [(numeric, fills["median"]),
                           (categorical, fills["own level"])],
    "median / proportional": [(numeric, fills["median"]),
                              (categorical, fills["proportional"])],
    "mean / mode": [(numeric, fills["mean"]), (categorical, fills["mode"])],
    "zero / mode": [(numeric, fills["zero"]), (categorical, fills["mode"])],
    "skew-aware": [(skewed, fills["median"]), (symmetric, fills["mean"]),
                   (categorical, fills["mode"])],
    "zero where none / median elsewhere": [
        (zero_means_none, fills["zero"]),
        ([c for c in numeric if c not in zero_means_none], fills["median"]),
        (categorical, fills["mode"])],
    # A label has no distance to measure, so these two fill numbers only.
    "KNN (k=5) / mode": [(numeric, fills["KNN (k=5)"]), (categorical, fills["mode"])],
    "iterative / mode": [(numeric, fills["iterative"]), (categorical, fills["mode"])],
}

plan_scores = {}

for name, plan in plans.items():
    print(f"Running: {name}", flush=True)

    plan_scores[name] = held_out_f1(
        partial(fill_missing, plan=plan),
        deduped,
        deduped_splits,
        numeric,
        categorical,
        scaler=StandardScaler()
    )

    print(f"Finished: {name}", flush=True)

Running: median / mode
Finished: median / mode
Running: median / own level
Finished: median / own level
Running: median / proportional
Finished: median / proportional
Running: mean / mode
Finished: mean / mode
Running: zero / mode
Finished: zero / mode
Running: skew-aware
Finished: skew-aware
Running: zero where none / median elsewhere
Finished: zero where none / median elsewhere
Running: KNN (k=5) / mode
Finished: KNN (k=5) / mode
Running: iterative / mode
Finished: iterative / mode


In [32]:
# Show the average F1 for each method in one table.
imputation_rows = []

for name, scores in plan_scores.items():
    imputation_rows.append({"Method": name, "Mean F1": scores.f1.mean()})

imputation_results = pd.DataFrame(imputation_rows)
imputation_results.round(4)

,Method,Mean F1
0,median / mode,0.6936
1,median / own level,0.6932
2,median / proportional,0.6926
3,mean / mode,0.6925
4,zero / mode,0.6941
5,skew-aware,0.6936
6,zero where none / median elsewhere,0.6938
7,KNN (k=5) / mode,0.6943
8,iterative / mode,0.6928


In [33]:
shipped = plan_scores["median / mode"]

# Paired: each arm minus the shipped arm on the SAME split, so the noise they
# share cancels and what is left is the difference between the arms.
paired = pd.DataFrame({
    name: {"dF1": (run.f1 - shipped.f1).mean(),
           "F1 SEM": sem(run.f1 - shipped.f1)}
    for name, run in plan_scores.items() if name != "median / mode"}).T
print(paired.round(4).to_string())

                                       dF1  F1 SEM
median / own level                 -0.0003  0.0009
median / proportional              -0.0009  0.0010
mean / mode                        -0.0010  0.0003
zero / mode                         0.0005  0.0007
skew-aware                          0.0000  0.0000
zero where none / median elsewhere  0.0002  0.0003
KNN (k=5) / mode                    0.0007  0.0006
iterative / mode                   -0.0008  0.0005


In [38]:
# The fill plan this section chose.
chosen = "KNN (k=5) / mode"
plan_scores[chosen].f1.mean()

np.float64(0.6942740215710885)

KNN (k=5) / mode had the highest average F1 here (0.6943), but the methods gave very similar scores. Just predicting a repeat purchase for everyone scored 0.6945, so KNN with standard scaling did not improve on that simple prediction. We use KNN for the next cleaning comparisons, but we'll check this choice again when we compare models.

In [39]:
def drop_incomplete(train, test, plan):
    """Drop the training rows carrying any missing value, then fill what the rest need."""
    complete = ~train.isna().any(axis=1)
    return fill_missing(train[complete], test, plan)


keep_rows = plan_scores[chosen].f1
drop_rows = held_out_f1(partial(drop_incomplete, plan=plans[chosen]),
                        deduped, deduped_splits, numeric, categorical,
                        scaler=StandardScaler()).f1
paired = keep_rows - drop_rows

print(f"impute the missing values  n={len(deduped):,}   F1 {keep_rows.mean():.4f}")
print(f"drop the rows    n={(~deduped.isna().any(axis=1)).sum():,}"
      f"   F1 {drop_rows.mean():.4f}")
print(f"paired difference {paired.mean():+.4f}"
      f" +/- {sem(paired):.4f}"
      f"   imputing wins on {(paired > 0).sum()} of {len(paired)} splits")

impute the missing values  n=6,469   F1 0.6943
drop the rows    n=4,488   F1 0.6830
paired difference +0.0113 +/- 0.0018   imputing wins on 18 of 20 splits


In [ ]:
spine = deduped.copy()

spine_splits = list(splits.split(repaired, repaired["repeat_purchase_90d"],
                                    groups=repaired["customer_id"]))
print(f"spine: {spine.shape[0]:,} rows x {spine.shape[1]} columns,"
      f" win rate {spine['repeat_purchase_90d'].mean():.4f},"
      f" {spine.isna().sum().sum():,} missing values left open")

spine: 6,469 rows x 38 columns, win rate 0.5597, 6,276 missing values left open


<b><font size="5" color="#E8800A">Outliers</font></b><br>

The boxplots done in the data exploration notebook showed a lot of outliers in the numeric columns. This section of the notebook is dedicated to exploring the best way to handle them. 

First we checked how the data would look if we used the IQR rule: $$ [Q_1 - 1.5(Q_3 - Q_1), Q_3 + 1.5(Q_3 - Q_1)] $$

The function below identifies which values are within the IQR interval represented above.

In [41]:
def outside_fences(frame, columns, rule="iqr", k=1.5):
    """True for rows with a named column beyond its fences.

    `rule="iqr"` draws the fences k IQRs beyond the quartiles, `rule="zscore"`
    k standard deviations from the mean. A missing value counts as INSIDE: a
    comparison with a missing value is False both ways, so without the `isna()`
    term every row carrying one would be called an outlier.
    """
    values = frame[columns]
    if rule == "iqr":
        q1, q3 = values.quantile(0.25), values.quantile(0.75)
        low, high = q1 - k * (q3 - q1), q3 + k * (q3 - q1)
    else:
        mean, sd = values.mean(), values.std(ddof=0)
        low, high = mean - k * sd, mean + k * sd
    within = values.ge(low) & values.le(high)
    return ~(within | values.isna()).all(axis=1)

In [42]:
# The detector defined above, with the 1.5 x IQR rule.
kept = ~outside_fences(spine, numeric, "iqr", k=1.5)
print(f"rows kept {kept.sum():,} of {len(spine):,}"
      f"  ({100 * (1 - kept.mean()):.2f}% removed)")
print(f"win rate  {spine['repeat_purchase_90d'].mean():.4f} ->"
      f" {spine.loc[kept, 'repeat_purchase_90d'].mean():.4f}")
print(f"win rate among the rows the rule DELETES:"
      f" {spine.loc[~kept, 'repeat_purchase_90d'].mean():.4f}")

rows kept 2,183 of 6,469  (66.25% removed)
win rate  0.5597 -> 0.4760
win rate among the rows the rule DELETES: 0.6024


In conclusion, if we applied the IQR rule, we would have to remove 66.25% of all rows. This would lower the win rate.

In [43]:
values = spine[numeric]
centred = values - values.mean()
scores_pop = centred / values.std(ddof=0)
scores_sample = centred / values.std(ddof=1)
print(f"|z| <= 3 with ddof=0 (numpy's default):"
      f" {(scores_pop.abs() <= 3).all(axis=1).sum():,} rows")
print(f"|z| <= 3 with ddof=1 (pandas' default):"
      f" {(scores_sample.abs() <= 3).all(axis=1).sum():,} rows")

|z| <= 3 with ddof=0 (numpy's default): 3,730 rows
|z| <= 3 with ddof=1 (pandas' default): 3,730 rows


Next we will test five different methods of how to handle the outliers.

In [44]:
def keep_everything(train, test):
    """The control arm: the frame, untouched."""
    return train, test


def drop_outside(train, test, columns, rule, k):
    """Drop the TRAINING rows the fences call outliers; the test rows stay."""
    return train[~outside_fences(train, columns, rule, k)], test


def winsorize(train, test, columns, lower=0.01, upper=0.99):
    """Clip the named columns to the given TRAINING quantiles."""
    low, high = train[columns].quantile(lower), train[columns].quantile(upper)
    train, test = train.copy(), test.copy()
    train[columns] = train[columns].clip(low, high, axis=1)
    test[columns] = test[columns].clip(low, high, axis=1)
    return train, test


def log1p_of(train, test, columns):
    """log1p the named columns, clipping negatives to zero first. It fits nothing."""
    train, test = train.copy(), test.copy()
    train[columns] = np.log1p(train[columns].clip(lower=0))
    test[columns] = np.log1p(test[columns].clip(lower=0))
    return train, test


def treat_then_fill(train, test, plan, treatment):
    """Apply the treatment under test, then fill what is left inside the split.

    Treating first means a fence or a quantile is never fitted over cells the
    fill invented.
    """
    return fill_missing(*treatment(train, test), plan)


# log1p goes on the long-tailed, non-negative columns only: right-skewed
# (skew above 1), no negative value, and more than ten distinct values, which
# leaves out the indicators and the small counts.
skew_ = spine[numeric].skew()
log_cols = [c for c in numeric
            if skew_[c] > 1 and spine[c].min() >= 0 and spine[c].nunique() > 10]
print(log_cols)

treatments = {
    "keep everything": keep_everything,
    "drop IQR 1.5x": partial(drop_outside, columns=numeric, rule="iqr", k=1.5),
    "drop |z| > 3": partial(drop_outside, columns=numeric, rule="zscore", k=3),
    "winsorize 1/99": partial(winsorize, columns=numeric, lower=0.01, upper=0.99),
    "log1p": partial(log1p_of, columns=log_cols),
}
treatment_scores = {
    name: held_out_f1(partial(treat_then_fill, plan=plans[chosen], treatment=fn),
                      spine, spine_splits, numeric, categorical,
                      scaler=StandardScaler()).f1
    for name, fn in treatments.items()}

# What each treatment does to the rows it is allowed to touch: the TRAINING
# half of the first split, missing values still open, because that is what the
# treatment sees.
train_rows, test_rows = spine_splits[0]
first_train = spine.iloc[train_rows]
treated = {name: fn(first_train, first_train.iloc[:0])[0]
           for name, fn in treatments.items()}
baseline = treatment_scores["keep everything"]

['Total a Pagar', 'Total Bruto', 'Total Liquido', 'Total Imp_', 'Total Portes', 'Total Desc_ Global', 'Total Desc_ Linha', 'Custo', 'N_ Detalhes', 'days_since_previous_purchase', 'prior_purchase_count', 'prior_total_spend', 'prior_mean_purchase_value', 'prior_std_purchase_value', 'prior_min_purchase_value', 'prior_max_purchase_value', 'prior_mean_gap_days', 'prior_std_gap_days', 'purchase_count_30d', 'spend_30d', 'purchase_count_90d', 'spend_90d', 'purchase_count_180d', 'spend_180d', 'purchase_count_365d', 'spend_365d']


In [45]:
per_treatment = {}
for name, frame in treated.items():
    scores = treatment_scores[name]
    difference = scores - baseline
    per_treatment[name] = {
        "rows kept": len(frame),
        "class rate": frame["repeat_purchase_90d"].mean(),
        "F1": scores.mean(),
        "vs keep": difference.mean(),
        "paired SEM": sem(difference),
        "beats keep on": f"{int((difference > 0).sum())} of {len(difference)}",
    }
outlier_board = pd.DataFrame.from_dict(per_treatment, orient="index")
print(outlier_board.round(4).to_string())

                 rows kept  class rate      F1  vs keep  paired SEM beats keep on
keep everything       5326      0.5747  0.6943   0.0000      0.0000       0 of 20
drop IQR 1.5x         1801      0.4975  0.6678  -0.0265      0.0066       3 of 20
drop |z| > 3          4102      0.5334  0.6930  -0.0013      0.0019       9 of 20
winsorize 1/99        5326      0.5747  0.6959   0.0017      0.0009      14 of 20
log1p                 5326      0.5747  0.7047   0.0104      0.0021      17 of 20


`log1p` scores highest, 0.0104 ± 0.0021 above keeping everything, and has an F1 score of 0.7047 which is the highest score between all treatments. Therefore, `log1p` is our chosen outlier treatment.

In [46]:
chosen_outlier_treatment = "log1p"

Using the chosen imputation plan and standard scaling on the same 20 grouped splits, `log1p` had the highest mean F1 (0.7047). Keeping all values unchanged scored 0.6943. This comparison keeps the validation rows the same for every treatment.

In [47]:
# Exploratory comparison. something we tried
# It changes the preprocessing order, row/group splitting and number of splits.
# These scores cannot isolate the effect of preprocessing order and are not
# used to choose the cleaning recipe.

# naive_frame, _ = fill_missing(spine, spine, plans[chosen])
# naive = {}
# for name in ["keep everything", "drop IQR 1.5x", "log1p"]:
#     frame = treatments[name](naive_frame, naive_frame.iloc[:0])[0]
#     matrix = design_matrix(frame, numeric, categorical)
#     train_X, test_X, train_y, test_y = train_test_split(
#         matrix, frame["repeat_purchase_90d"], test_size=0.2, stratify=frame["repeat_purchase_90d"],
#         random_state=RANDOM_STATE)
#     model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
#     model.fit(train_X, train_y)
#     naive[name] = f1_score(test_y, model.predict(test_X))
#
# pd.DataFrame({"treat first, then split": naive,
#               "split first, 20 splits":
#                   {k: treatment_scores[k].mean() for k in naive}}).round(4)


<b><font size="5" color="#E8800A">Scaling</font></b><br>

In [48]:
scaler_candidates = {
    "unscaled": None,
    "standard": StandardScaler(),
    "min-max": MinMaxScaler(),
    "robust": RobustScaler(),
}
recipe_frame = partial(treat_then_fill, plan=plans[chosen],
                       treatment=treatments[chosen_outlier_treatment])
scaler_runs = {
    name: held_out_f1(recipe_frame, spine, spine_splits, numeric, categorical,
                      scaler=scaler)
    for name, scaler in scaler_candidates.items()
}
scaler_board = pd.DataFrame({
    "F1": {name: run.f1.mean() for name, run in scaler_runs.items()},
    "SEM": {name: sem(run.f1) for name, run in scaler_runs.items()},
    "iterations": {name: run.iterations.mean()
                   for name, run in scaler_runs.items()},
})
print(scaler_board.round({"F1": 4, "SEM": 4, "iterations": 0}).to_string())

              F1     SEM  iterations
unscaled  0.7055  0.0146      1000.0
standard  0.7047  0.0142        53.0
min-max   0.7109  0.0141        58.0
robust    0.7094  0.0140        97.0


With KNN imputation and log1p, min-max had the highest average F1 (0.7109), followed by robust scaling (0.7094). This is about 0.0164 above the always-repeat score (0.6945), so the improvement is fairly small.

The model without scaling hit the 1,000-iteration limit in all 20 splits, so it may have stopped before finishing the fit. We used standard scaling for the earlier cleaning comparisons to keep the conditions the same. We'll check these choices again when we compare models.

<b><font size="6" color="#E8800A">Log</font></b><br>

In [49]:
log = CleaningLog("train")
frame = train_copy.copy()   # the same-row decisions above are already in train_copy


def changed(col):
    a, b = train_copy[col], train_df.loc[train_copy.index, col]
    return int((a.ne(b) & ~(a.isna() & b.isna())).sum())


log.record("purchase_date", "converted from text to datetime ('%Y-%m-%d')",
           "the dates arrive as 'yyyy-mm-dd' strings, which cannot be sorted by time or "
           "subtracted; as datetime values they give the order and the intervals the "
           "dictionary says are preserved, and the column stays out of the one-hot encoder",
           int(train_df["purchase_date"].notna().sum()),
           carries={"applied": True, "dtype": "datetime64", "format": "%Y-%m-%d"})

log.record("all feature columns", "drop rows empty in every feature",
           "65 rows have no value in any numeric or categorical column, so there "
           "is nothing to compute or learn from",
           len(train_df) - len(train_copy), carries={"applied": True})

log.record("prior_mean_purchase_value",
           "recomputed as prior_total_spend / prior_purchase_count",
           "the stored mean was missing or negative for customers with a history, "
           "and the total and the count are on the same row",
           changed("prior_mean_purchase_value"), carries={"applied": True})

log.record("prior_mean_gap_days",
           "computed from customer_tenure_days and the purchase count",
           "(tenure - days since previous) / (count - 1) reproduces the stored "
           "mean gap wherever both exist",
           changed("prior_mean_gap_days"), carries={"applied": True})

log.record("spend_365d",
           "inconsistent values blanked; filled from prior_total_spend or 0 where "
           "the window holds all of the history or none of it",
           "a 365-day spend cannot be below the 180-day spend or above the total spend",
           changed("spend_365d"), carries={"applied": True})

log.record("prior_std_gap_days, prior_std_purchase_value",
           "0 for one gap or one purchase",
           "zero is used as a modelling convention for one prior purchase "
           "or one historical gap",
           changed("prior_std_gap_days") + changed("prior_std_purchase_value"),
           carries={"applied": True})

nulls_before = frame.isna().sum().sum()
frame = fix_invalid(frame, invalid_cols, invalid_errors, np.nan)
clip = invalid_board["clip to zero"]
log.record(", ".join(invalid_cols), f"invalid values -> NaN ('{chosen_invalid_fix}')",
           "a day count or a window spend cannot be negative; the -4 in the two "
           "day columns is a placeholder code; the clipping comparison against "
           f"blanking was {clip['vs blank']:+.4f} +/- {clip['paired SEM']:.4f}",
           int(frame.isna().sum().sum() - nulls_before),
           carries={"treatment": "blank", "becomes": None, "columns": list(invalid_cols)})

keep = tiebreak_board["keep every row"]
one = tiebreak_board["earliest ID"]
log.record("customer_id", chosen_tiebreak,
           f"keeping every row scored {keep['vs earliest']:+.4f} +/- {keep['paired SEM']:.4f} "
           f"against the earliest-ID rule; one row per customer keeps {one['rows']:,} of "
           f"{len(frame):,} rows and moves the win rate from {frame['repeat_purchase_90d'].mean():.4f} "
           f"to {one['win rate']:.4f}; customers are kept apart by splitting on customer_id",
           0, carries={"key": "customer_id", "keep": chosen_tiebreak})

log.record(f"{len(numeric)} numeric and {len(categorical)} categorical columns",
           f"missing values LEFT OPEN; the '{chosen}' plan goes to later weeks",
           "a median or a mode computed here would be computed over the rows each "
           "later week holds out to score on; the plan is applied inside each split",
           int(frame.isna().sum().sum()),
           carries={"applied": False, "plan": chosen,
                    "numeric": numeric, "categorical": categorical})

outlier_columns = list(numeric)
if chosen_outlier_treatment == "log1p":
    outlier_columns = list(log_cols)
elif chosen_outlier_treatment == "keep everything":
    outlier_columns = []

row = outlier_board.loc[chosen_outlier_treatment]
log.record("numeric columns", f"outlier treatment: {chosen_outlier_treatment}",
           f"five treatments compared with standard scaling on {len(spine_splits)} "
           f"paired splits of identical held-out "
           f"customers; {chosen_outlier_treatment} scored F1 {row['F1']:.4f}, "
           f"{row['vs keep']:+.4f} +/- {row['paired SEM']:.4f} against keeping everything",
           0, carries={"applied": False, "treatment": chosen_outlier_treatment,
                       "columns": outlier_columns})

record = pd.DataFrame(log.records())
print(record.drop(columns="reason").to_string(index=False))
print(f"\n{len(train_df):,} rows in -> {len(frame):,} rows out")

Path("../logs").mkdir(exist_ok=True)
log.to_json("../logs/train_cleaning_log.json")
print(f"log written: {len(log)} decisions,"
      f" {sum(step.carries is not None for step in log.steps)} carrying a plan")

                                      column                                                                                                                  action  rows_affected
                               purchase_date                                                                            converted from text to datetime ('%Y-%m-%d')           6469
                         all feature columns                                                                                        drop rows empty in every feature             65
                   prior_mean_purchase_value                                                                  recomputed as prior_total_spend / prior_purchase_count           1244
                         prior_mean_gap_days                                                               computed from customer_tenure_days and the purchase count            607
                                  spend_365d inconsistent values blanked; filled from prior_total_sp

In [50]:
cleaned = frame.copy()

Path("../data/interim").mkdir(parents=True, exist_ok=True)
cleaned.to_csv("../data/interim/train_cleaned.csv")
print(f"written: {cleaned.shape[0]:,} rows x {cleaned.shape[1]} columns,"
      f" {cleaned.isna().sum().sum():,} missing values")

written: 6,469 rows x 38 columns, 6,276 missing values
